In [1]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import ta  # Technical Analysis library (pip install ta)

df = pd.read_csv('../data/processed/merged/merged_data.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df.head()

,Date,gold_close,gold_high,gold_low,gold_open,gold_vol,dxy_close,vix_close,yield_close,sp500_close,oil_close
0,2015-01-02,1186.000000,1194.500000,1169.500000,1184.000000,138,91.080002,17.790001,2.123,2058.199951,52.689999
1,2015-01-05,1203.900024,1206.900024,1180.099976,1180.300049,470,91.379997,19.920000,2.039,2020.579956,50.040001
2,2015-01-06,1219.300049,1220.000000,1203.500000,1203.500000,97,91.500000,21.120001,1.963,2002.609985,47.930000
3,2015-01-07,1210.599976,1219.199951,1210.599976,1219.199951,29,91.889999,19.309999,1.954,2025.900024,48.650002
4,2015-01-08,1208.400024,1215.699951,1206.300049,1207.000000,92,92.370003,17.010000,2.016,2062.139893,48.790001


# Feature Engineering

ในส่วนนี้จะสร้างฟีเจอร์ 3 กลุ่มหลัก ได้แก่
- Lag Features (ย้อนหลัง 1, 5, 10 วัน)
- Technical Indicators (RSI, MACD, Bollinger Bands, ATR)
- Macro %change (DXY, VIX, Yield, SP500, Oil)

และสร้าง Target Variable สำหรับงาน Regression และ Classification

## 1. Macro %Change Features

สร้างฟีเจอร์ %change ของ DXY, VIX, Yield, SP500, Oil

In [2]:
# สร้าง pct_change ของทุกตัวก่อน เพื่อนำไปทำ Lag ต่อ
for col in ['gold_close','dxy_close','vix_close','yield_close','sp500_close','oil_close']:
    df[f'{col}_pct'] = df[col].pct_change() * 100

df[[c for c in df.columns if '_pct' in c]].head()

,gold_close_pct,dxy_close_pct,vix_close_pct,yield_close_pct,sp500_close_pct,oil_close_pct
0,NaN,NaN,NaN,NaN,NaN,NaN
1,1.509277,0.329376,11.973013,-3.956659,-1.827811,-5.029413
2,1.279178,0.131323,6.024100,-3.727316,-0.889347,-4.216628
3,-0.713530,0.426229,-8.570082,-0.458485,1.162984,1.502193
4,-0.181724,0.522367,-11.910923,3.172980,1.788828,0.287769


## 2. Lag Features

สร้างฟีเจอร์ราคาย้อนหลัง (lag) ของแต่ละสินทรัพย์ ได้แก่ gold, dxy, vix, yield, sp500, oil โดยใช้ช่วงเวลา 1, 5, 10 วัน

In [3]:
# สร้าง lag ทุกตัวก่อน
for col in ['gold_close','dxy_close','vix_close','yield_close','sp500_close','oil_close']:
    for lag in [1, 5, 10]:
        df[f'{col}_pct_lag{lag}'] = df[f'{col}_pct'].shift(lag)

df.head()

,Date,gold_close,gold_high,gold_low,gold_open,gold_vol,dxy_close,vix_close,yield_close,sp500_close,...,vix_close_pct_lag10,yield_close_pct_lag1,yield_close_pct_lag5,yield_close_pct_lag10,sp500_close_pct_lag1,sp500_close_pct_lag5,sp500_close_pct_lag10,oil_close_pct_lag1,oil_close_pct_lag5,oil_close_pct_lag10
0,2015-01-02,1186.000000,1194.500000,1169.500000,1184.000000,138,91.080002,17.790001,2.123,2058.199951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-05,1203.900024,1206.900024,1180.099976,1180.300049,470,91.379997,19.920000,2.039,2020.579956,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-06,1219.300049,1220.000000,1203.500000,1203.500000,97,91.500000,21.120001,1.963,2002.609985,...,NaN,-3.956659,NaN,NaN,-1.827811,NaN,NaN,-5.029413,NaN,NaN
3,2015-01-07,1210.599976,1219.199951,1210.599976,1219.199951,29,91.889999,19.309999,1.954,2025.900024,...,NaN,-3.727316,NaN,NaN,-0.889347,NaN,NaN,-4.216628,NaN,NaN
4,2015-01-08,1208.400024,1215.699951,1206.300049,1207.000000,92,92.370003,17.010000,2.016,2062.139893,...,NaN,-0.458485,NaN,NaN,1.162984,NaN,NaN,1.502193,NaN,NaN


## 3. Technical Indicators

สร้างฟีเจอร์ทางเทคนิค เช่น RSI, MACD, Bollinger Bands, ATR จากราคาทองคำ (gold_close)

In [4]:

import ta 

df['gold_rsi'] = ta.momentum.RSIIndicator(df['gold_close'], window=14).rsi()
df['gold_macd'] = ta.trend.MACD(df['gold_close']).macd()
df['gold_macd_diff'] = ta.trend.MACD(df['gold_close']).macd_diff()
bb = ta.volatility.BollingerBands(df['gold_close'], window=20)
df['gold_bb_pband'] = bb.bollinger_pband()
df['gold_atr'] = ta.volatility.AverageTrueRange(df['gold_high'], df['gold_low'], df['gold_close']).average_true_range()

df[[c for c in df.columns if 'rsi' in c or 'macd' in c or 'bb' in c or 'atr' in c]].head(20)

,gold_rsi,gold_macd,gold_macd_diff,gold_bb_pband,gold_atr
0,NaN,NaN,NaN,NaN,0.000000
1,NaN,NaN,NaN,NaN,0.000000
2,NaN,NaN,NaN,NaN,0.000000
3,NaN,NaN,NaN,NaN,0.000000
4,NaN,NaN,NaN,NaN,0.000000
5,NaN,NaN,NaN,NaN,0.000000
6,NaN,NaN,NaN,NaN,0.000000
7,NaN,NaN,NaN,NaN,0.000000
8,NaN,NaN,NaN,NaN,0.000000
9,NaN,NaN,NaN,NaN,0.000000


## 4. Log Transform

ข้อมูล Volume ของทองคำ มักจะมีปัญหาเรื่อง Contract Rollover (การเปลี่ยนสัญญา) ทำให้เกิด Spike สูงโดดในบางวัน

In [5]:
df['gold_vol_log'] = np.log1p(df['gold_vol'])

## 5. Target Variables

สร้างตัวแปรเป้าหมายสำหรับงาน Regression (return_tomorrow) และ Classification (direction)

In [6]:
# Regression target: gold return tomorrow
df['return_tomorrow'] = df['gold_close'].pct_change().shift(-1) * 100

# Classification target: direction
def label(ret):
    if pd.isna(ret):
        return np.nan
    if ret > 0.5:
        return 1    # Up
    elif ret < -0.5:
        return -1   # Down
    else:
        return 0    # Side
df['direction'] = df['return_tomorrow'].apply(label)
df[['return_tomorrow', 'direction']].head(10)

,return_tomorrow,direction
0,1.509277,1.0
1,1.279178,1.0
2,-0.713530,-1.0
3,-0.181724,0.0
4,0.628929,1.0
5,1.373351,1.0
6,0.129804,0.0
7,0.008100,0.0
8,2.454628,1.0
9,0.964661,1.0


In [7]:
df = df.dropna().reset_index(drop=True)
df.to_csv('../data/processed/featured/features.csv', index=False)
print(f"Final shape: {df.shape}")
print(f"Features: {len(df.columns)} columns")

Final shape: (2791, 43)
Features: 43 columns
